In [4]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(
    SparkSession.builder
        .appName("Python Spark SQL basic example")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.driver.memory", "4g")
        .config("spark.executor.memory", "4g")
        .config("spark.sql.codegen.wholeStage", "false")
).getOrCreate()


23/10/16 01:53:17 WARN Utils: Your hostname, codespaces-7c531b resolves to a loopback address: 127.0.0.1; using 172.16.5.4 instead (on interface eth0)
23/10/16 01:53:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-06204671-218b-43cd-a2ec-27e97e40ebe6;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 294ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0 

In [5]:
%%sh
ls -alh /tmp/data


total 28K
drwxr-xr-x+  7 root root 4.0K Oct 16 01:46 .
drwxr-xrwt+ 16 root root 4.0K Oct 16 01:53 ..
drwxr-xr-x+  2 root root 4.0K Oct 16 01:46 csv
drwxr-xr-x+  2 root root 4.0K Oct 16 01:46 delta
drwxr-xr-x+  2 root root 4.0K Oct 16 01:46 json
drwxr-xr-x+  2 root root 4.0K Oct 16 01:46 orc
drwxr-xr-x+  2 root root 4.0K Oct 16 01:46 parquet


In [6]:
from pyspark.sql.functions import input_file_name, substring_index, regexp_extract, regexp_replace

def get_datatype(path: str) -> str:
    files = spark.read.text(path + "/*")\
        .select(
            input_file_name().alias('full_path'),
            regexp_replace(input_file_name(), "^" + path + "/", "./").alias('relative_path'), 
            substring_index(input_file_name(), "/", -1).alias('file_name'), 
            substring_index(input_file_name(), ".", -1).alias('extension')
        )
    
    if files.filter(files.relative_path.contains("_delta_log")).count() >= 1:
        return "delta"
    else:
        return files.groupBy("extension").count().first()[0]


In [8]:
import os 

for dir in os.listdir("/tmp/data"):
    print(dir, get_datatype(f"/tmp/data/{dir}"))


csv csv
delta parquet
orc orc
parquet parquet
json json


In [9]:
get_datatype("/tmp/standalone_iris.csv")


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/tmp/standalone_iris.csv/*.